In [1]:
# Libraries & Imports
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.1)
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
HORIZONS = [1, 3, 7]
np.random.seed(RANDOM_STATE)

PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('../reports/figures')
REPORTS_DIR = Path('../reports')
MODELS_DIR = Path('../models')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# SHAP is used to verify the models. Import it if present, if not, the notebook stays runnable by falling back to permutation importance.
try:
    import shap
    HAS_SHAP = True
    print(f'shap {shap.__version__} available - using SHAP for feature attribution.')
except ImportError:
    HAS_SHAP = False
    print('shap not installed - will fall back to permutation importance.')

print('All libraries loaded.')

shap 0.52.0 available - using SHAP for feature attribution.
All libraries loaded.


### Results

**Goal of this notebook:** take the model and horizon candidates that were selected in `03_modelling.ipynb` on the 2024 validation year, retrain them on all pre-test data (2016-2024), and evaluate them **once** on the held-out sets - the 2025 calendar year (primary, headline metrics) and the Jan-Apr 2026 recency window (reported separately).

This notebook answers:
1. Do the validation-selected models hold their skill on data from a year they never saw, or was the validation performance optimistic? (the **generalisation gap**)
2. How large is the skill over the persistence and climatology baselines on truly held-out data, per horizon?
3. Does the unified model still generalise **per city** on the test year, including the alpine Yundola and the coastal stations?
4. **Which features actually drive the forecasts** (SHAP), and do they match the physical predictors the EDA and feature-engineering notebooks bet on (pressure, dew point, soil temperature, the autoregressive lags)?
5. What would it take to run this operationally - live data, retraining cadence, a forecast UI - and what are the honest limitations of this project approach?

#### The One-shot Test Discipline

The test sets were created in notebook 02 and deliberately left unopened through notebook 03. Every model family, hyperparameter and decision threshold was fixed only on 2024 validation data. The reasoning follows Géron's warning that each time each time a test set is used to *make a choice*, it stops being an unbiased estimate of generalisation error and the reported skill drifts upward [2]. So here the test data is used exactly once, for measurement only - no tuning, no threshold re-fitting, no model swapping based on what the results are.


#### Pre-registered Candidates Carried Over From Notebook 03

These selections were made on **validation only** and are now frozen. The chosen model at each horizon is the primary model and the others are retrained alongside it purely so the test table mirrors the validation table and the gap is visualized uniformly rather than cherry-picked.

**Temperature regression (primary metric: RMSE)**

| Horizon | Champion | Why it won on validation |
|---|---|---|
| 1-day | HistGradientBoosting | Best val RMSE; today's synoptic state is strongly informative at short range. |
| 3-day | Ridge | Trees start fitting noise as the signal weakens; shrinkage generalises better. |
| 7-day | Ridge | Same bias-variance argument, more pronounced at long range. |

**Rain classification (primary metric: PR-AUC; decision threshold fixed on validation)**

| Horizon | Champion | Why it won on validation |
|---|---|---|
| 1-day | HistGradientBoosting | Best PR-AUC / AUC-ROC; captures humidity x pressure x cloud conjunctions. |
| 3-day | RandomForest | Edges ahead on PR-AUC where the signal is weak. |
| 7-day | RandomForest | Same; all three families sit within noise of each other here. |

The hyperparameters below are exactly the ones tuned on validation in notebook 03. The rain decision thresholds are reloaded from the notebook-03 run log, so no threshold is fit on test.

In [2]:
print('Environment (recorded for reproducibility):')
print(f' -python: {sys.version.split()[0]} ({platform.system()})')
print(f' -numpy: {np.__version__}')
print(f' -pandas: {pd.__version__}')
print(f' -scikit-learn: {sklearn.__version__}')
print(f' -random_state: {RANDOM_STATE}')

# Validation-tuned hyperparameters carried over from notebook 03.
RIDGE_PARAMS = {'alpha': 0.1}
HGB_REG_PARAMS = {'learning_rate': 0.03, 'max_leaf_nodes': 31,
                  'min_samples_leaf': 20, 'l2_regularization': 0.0, 'max_iter': 400}
RF_REG_PARAMS = {'n_estimators': 200, 'max_depth': 20, 'min_samples_leaf': 5}

LOGIT_PARAMS = {'C': 1.0, 'max_iter': 1000}
RF_CLF_PARAMS = {'n_estimators': 300, 'max_depth': 16, 'min_samples_leaf': 5}
HGB_CLF_PARAMS = {'learning_rate': 0.03, 'max_leaf_nodes': 31,
                  'min_samples_leaf': 30, 'l2_regularization': 1.0, 'max_iter': 400}

# Pre-registered champions (chosen on validation in notebook 03)
TEMP_SELECTION = {1: 'HistGradientBoosting', 3: 'Ridge', 7: 'Ridge'}
RAIN_SELECTION = {1: 'HistGradientBoosting', 3: 'RandomForest', 7: 'RandomForest'}

print('\nPre-registered models:')
print(' temperature:', TEMP_SELECTION)
print(' rain:', RAIN_SELECTION)

Environment (recorded for reproducibility):
 -python: 3.13.9 (Windows)
 -numpy: 2.3.5
 -pandas: 2.3.3
 -scikit-learn: 1.7.2
 -random_state: 42

Pre-registered models:
 temperature: {1: 'HistGradientBoosting', 3: 'Ridge', 7: 'Ridge'}
 rain: {1: 'HistGradientBoosting', 3: 'RandomForest', 7: 'RandomForest'}


#### Loading Data

Next step is to load all four splits - including the test parquets that were not used in notebook 03. The fitted `StandardScaler` and the feature/target column lists come from notebook 02 unchanged, so the test data goes through the identical preprocessing the models were trained under. The notebook-03 run log and selection tables are also reloaded: the validation numbers (for the generalisation-gap comparison) and the validation-tuned rain thresholds both come from there.

In [3]:
train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
val = pd.read_parquet(PROCESSED_DIR / 'val.parquet')
test_2025 = pd.read_parquet(PROCESSED_DIR / 'test_2025.parquet')
test_2026 = pd.read_parquet(PROCESSED_DIR / 'test_2026.parquet')
scaler = joblib.load(PROCESSED_DIR / 'scaler.joblib')

with open(PROCESSED_DIR / 'feature_columns.json', encoding='utf-8') as f:
    col_spec = json.load(f)
feature_cols = col_spec['feature_cols']
scaled_cols = col_spec['scaled_cols']

# Notebook-03 outputs: full run log + the two validation selection tables.
val_results = pd.read_csv(REPORTS_DIR / 'modelling_results.csv')
sel_temp = pd.read_csv(REPORTS_DIR / 'selection_temperature.csv')
sel_rain = pd.read_csv(REPORTS_DIR / 'selection_rain.csv')

for name, frame in [('train', train), ('val', val), ('test_2025', test_2025), ('test_2026', test_2026)]:
    print(f'{name:10s}: {frame.shape} - {frame.date.min().date()} - {frame.date.max().date()}')
print(f'\nfeatures: {len(feature_cols)};  targets: {col_spec["target_cols"]}')

train     : (29150, 84) - 2016-01-08 - 2023-12-31
val       : (3660, 84) - 2024-01-01 - 2024-12-31
test_2025 : (3650, 84) - 2025-01-01 - 2025-12-31
test_2026 : (1140, 84) - 2026-01-01 - 2026-04-24

features: 76;  targets: ['tavg_target_1d', 'rain_target_1d', 'tavg_target_3d', 'rain_target_3d', 'tavg_target_7d', 'rain_target_7d']


#### Building the Train and Validation Fitting Set

For the final models, training uses the pooled 2016–2024 set (train + validation). Validation has already served its purpose (model and threshold selection), so retaining it separately offers no further benefit. Pooling maximizes pre-test history for the frozen final fit, consistent with standard best practice. 

The train/test boundary is unchanged: test data still begins strictly after 2024, preserving the no-leakage guarantee.Baseline values are converted back to physical units by inverting the stored train-fit scaler on the relevant column (`raw = scaled * scale_ + mean_`). Climatology normals are computed over the same pooled 2016–2024 period and never from test data.

In [4]:
trainval = pd.concat([train, val], ignore_index=True)

def inverse_col(frame, col):
    "Recover a single column in original units from the train-fit StandardScaler."
    idx = scaled_cols.index(col)
    return frame[col].to_numpy() * scaler.scale_[idx] + scaler.mean_[idx]

for frame in (trainval, test_2025, test_2026):
    frame['tavg_now'] = inverse_col(frame, 'tavg')
    frame['rain_now'] = inverse_col(frame, 'rain')

X_trainval = trainval[feature_cols]
X_test_2025 = test_2025[feature_cols]
X_test_2026 = test_2026[feature_cols]

print(f'Combined fitting set (train+val): {trainval.shape}  '
      f'{trainval.date.min().date()} - {trainval.date.max().date()}')

# Leakage guards
order_2025_ok = trainval['date'].max() < test_2025['date'].min()
order_2026_ok = test_2025['date'].max() < test_2026['date'].min()
trainval_nan = int(trainval[feature_cols].isna().sum().sum())
test2025_nan = int(test_2025[feature_cols].isna().sum().sum())

print('\nLeakage / integrity guards:')
print(f"  train+val ends {trainval['date'].max().date()} < 2025 test starts "
      f"{test_2025['date'].min().date()} - {order_2025_ok}")
print(f"  2025 test ends {test_2025['date'].max().date()} < 2026 recency starts "
      f"{test_2026['date'].min().date()} - {order_2026_ok}")
print(f"  NaNs in train+val features: {trainval_nan} - {trainval_nan == 0}")
print(f"  NaNs in 2025 test features: {test2025_nan} - {test2025_nan == 0}")

assert order_2025_ok, 'train+val must precede 2025 test'
assert order_2026_ok, '2025 test must precede 2026 recency'
assert trainval_nan == 0, 'no NaNs allowed in fitting features'
assert test2025_nan == 0, 'no NaNs allowed in 2025 features'
print('All guards passed.')

# Climatology lookup fit on pre-test years only
clim_table = (trainval.assign(doy=trainval.date.dt.dayofyear)
              .groupby(['city', 'doy'])['tavg_now'].mean())
city_mean_tavg = trainval.groupby('city')['tavg_now'].mean()

def climatology_predict(frame, horizon):
    "Seasonal-normal forecast for the target day (date + horizon), per city."
    target_doy = (frame['date'] + pd.Timedelta(days=horizon)).dt.dayofyear
    keys = zip(frame['city'].to_numpy(), target_doy.to_numpy())
    return np.array([clim_table.get(k, city_mean_tavg[k[0]]) for k in keys])

print(f'Climatology table: {clim_table.shape[0]} (city, day-of-year) cells from 2016-2024')

Combined fitting set (train+val): (32810, 86)  2016-01-08 - 2024-12-31

Leakage / integrity guards:
  train+val ends 2024-12-31 < 2025 test starts 2025-01-01 - True
  2025 test ends 2025-12-31 < 2026 recency starts 2026-01-01 - True
  NaNs in train+val features: 0 - True
  NaNs in 2025 test features: 0 - True
All guards passed.
Climatology table: 3660 (city, day-of-year) cells from 2016-2024


In [5]:
def make_reg(name):
    "Construct a fresh regressor with its validation-tuned hyperparameters."
    if name == 'Ridge':
        return Ridge(**RIDGE_PARAMS)
    if name == 'RandomForest':
        return RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE, **RF_REG_PARAMS)
    if name == 'HistGradientBoosting':
        return HistGradientBoostingRegressor(random_state=RANDOM_STATE, **HGB_REG_PARAMS)
    raise ValueError(name)

def regression_metrics(y_true, pred, persist_mae, clim_mae):
    "RMSE, MAE and skill over both reference baselines."
    mae = mean_absolute_error(y_true, pred)
    return {
        'rmse': np.sqrt(mean_squared_error(y_true, pred)),
        'mae': mae,
        'skill_vs_persist': 1 - mae / persist_mae,
        'skill_vs_clim': 1 - mae / clim_mae,
    }

reg_families = ['Ridge', 'RandomForest', 'HistGradientBoosting']
reg_rows = []
final_reg_models = {} #(selected model name, horizon): fitted model, for SHAP/saving
reg_test_pred = {}  # (model, horizon, split): predictions

print('Retraining temperature models on 2016-2024 and scoring on held-out sets...\n')
for h in HORIZONS:
    y_tv = trainval[f'tavg_target_{h}d'].to_numpy()
    targets = {'2025': (test_2025, test_2025[f'tavg_target_{h}d'].to_numpy()),
               '2026': (test_2026, test_2026[f'tavg_target_{h}d'].to_numpy())}

    # Baselines on each test split
    for split, (frame, y_true) in targets.items():
        persist = frame['tavg_now'].to_numpy()
        clim = climatology_predict(frame, h)
        p_mae = mean_absolute_error(y_true, persist)
        c_mae = mean_absolute_error(y_true, clim)
        for bname, bpred, smae in [('Persistence', persist, p_mae), ('Climatology', clim, c_mae)]:
            m = regression_metrics(y_true, bpred, p_mae, c_mae)
            reg_rows.append({'split': split, 'horizon': h, 'model': bname, **m})

    # Learned families
    for name in reg_families:
        t0 = time.time()
        model = make_reg(name).fit(X_trainval, y_tv)
        fit_sec = time.time() - t0
        for split, (frame, y_true) in targets.items():
            X = frame[feature_cols]
            pred = model.predict(X)
            reg_test_pred[(name, h, split)] = pred
            persist = frame['tavg_now'].to_numpy()
            clim = climatology_predict(frame, h)
            m = regression_metrics(y_true, pred, mean_absolute_error(y_true, persist), mean_absolute_error(y_true, clim))
            reg_rows.append({'split': split, 'horizon': h, 'model': name, **m})
        if name == TEMP_SELECTION[h]:
            final_reg_models[(name, h)] = model
            print(f'h={h}d selected model: {name:20s} fit {fit_sec:4.1f}s')

reg_results = pd.DataFrame(reg_rows)
print('\nTemperature evaluation complete.')

Retraining temperature models on 2016-2024 and scoring on held-out sets...

h=1d selected model: HistGradientBoosting fit  6.0s
h=3d selected model: Ridge                fit  0.0s
h=7d selected model: Ridge                fit  0.0s

Temperature evaluation complete.


In [8]:
order = ['Persistence', 'Climatology', 'Ridge', 'RandomForest', 'HistGradientBoosting']
t2025 = reg_results[reg_results.split == '2025']

print('TEST 2025 - RMSE (degrees C) by model x horizon:')
print(t2025.pivot_table(index='model', columns='horizon', values='rmse').reindex(order).round(3).to_string())
print('\nTEST 2025 - MAE (degrees C):')
print(t2025.pivot_table(index='model', columns='horizon', values='mae').reindex(order).round(3).to_string())
print('\nTEST 2025 - Skill vs persistence (MAE-based; >0 beats persistence):')
print(t2025.pivot_table(index='model', columns='horizon', values='skill_vs_persist').reindex(order[1:]).round(3).to_string())
print('\nTEST 2025 - Skill vs climatology (>0 beats the seasonal normal):')
print(t2025.pivot_table(index='model', columns='horizon', values='skill_vs_clim').reindex(order[2:]).round(3).to_string())

print('\n\nHeadline RMSE on 2025:')
for h in HORIZONS:
    selected = TEMP_SELECTION[h]
    r = t2025[(t2025.horizon == h) & (t2025.model == selected)].iloc[0]
    print(f'  {h}-day: {selected:20s} RMSE={r.rmse:.3f}  MAE={r.mae:.3f}  '
          f'skill(persist)={r.skill_vs_persist:+.2f}  skill(clim)={r.skill_vs_clim:+.2f}')

TEST 2025 - RMSE (degrees C) by model x horizon:
horizon                   1      3      7
model                                    
Persistence           1.918  3.631  4.689
Climatology           3.703  3.710  3.741
Ridge                 1.675  3.057  3.520
RandomForest          1.708  3.248  3.994
HistGradientBoosting  1.677  3.232  3.857

TEST 2025 - MAE (degrees C):
horizon                   1      3      7
model                                    
Persistence           1.445  2.821  3.647
Climatology           2.847  2.857  2.878
Ridge                 1.265  2.332  2.671
RandomForest          1.278  2.493  2.963
HistGradientBoosting  1.250  2.476  2.872

TEST 2025 - Skill vs persistence (MAE-based; >0 beats persistence):
horizon                   1      3      7
model                                    
Climatology          -0.970 -0.013  0.211
Ridge                 0.125  0.173  0.268
RandomForest          0.116  0.116  0.188
HistGradientBoosting  0.135  0.122  0.213

TEST 2025 -

At 1-day, all three models beat both baselines: Ridge and HistGradientBoosting are essentially tied for lowest RMSE (1.675 / 1.677°C). As horizon increases, skill vs. climatology falls (Ridge and RandomForest), while skill vs. persistence rises, consistent with persistence's RMSE growing fastest and climatology's staying near-flat (3.70→3.74°C). Ridge has the lowest RMSE at 3-day (3.057) and 7-day (3.520), overtaking the tree-based models as horizon lengthens. All models beat persistence at every horizon, but by 7-day only Ridge and HistGradientBoosting still (marginally) beat climatology.